# Coil Slip Register — Data Analysis

**Purpose:** Is Excel file ke andar jo actual data hai, sirf usi par analysis karna hai.  
Koi fake/sample production data add nahi kiya gaya.

### Analysis flow
1. Excel data load
2. Data structure aur data quality check
3. Basic KPIs
4. Scrap analysis
5. Coil type analysis
6. Total Size analysis
7. Date/month trend
8. Useful charts
9. Business findings

> **Important:** Source Excel ko overwrite/change nahi kiya ja raha. Date values ko analysis ke liye parse kiya gaya hai, lekin original values preserve rehti hain.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

FILE = "coil_slip_file.xlsx"

# Header Excel sheet ke row 4 mein hai, isliye skiprows=3
df = pd.read_excel(FILE, sheet_name="Sheet1", skiprows=3)

df.head()


In [ ]:
# Column names ko clean kiya ja raha hai.
# Ye source Excel ko change nahi karta; sirf analysis DataFrame ke labels clean karta hai.

df.columns = [
    "Date",
    "Total Size",
    "Weight",
    "Seera",
    "Side Cutting",
    "Coil",
    "Scrap",
    "Total Weight"
]

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.info()


## 1. Data Quality Check

Yahan hum pehle dekh rahe hain:
- missing values
- duplicate rows
- numeric columns
- date values
- calculation consistency

**Note:** Excel mein kuch dates actual Excel-date format mein hain aur kuch text format mein. Isliye date analysis ke liye separate parsed date banayenge. Original `Date` column ko touch nahi karenge.


In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:
# Duplicate complete rows
print("Duplicate rows:", df.duplicated().sum())

# Numeric columns
numeric_cols = ["Total Size", "Weight", "Seera", "Side Cutting", "Scrap", "Total Weight"]
df[numeric_cols].describe().T


In [ ]:
# Date parsing for analysis only.
# Text dates like 15/6/2026 are read day-first.
# Existing Excel datetime values remain as they are stored.

df["Analysis Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)

print("Unparsed dates:", df["Analysis Date"].isna().sum())
print(df[["Date", "Analysis Date"]].head(20))


## 2. Calculation Validation

Workbook ke data mein `Total Weight` ko check karenge:

**Weight + Scrap = Total Weight**

Agar difference zero hai to record internally consistent hai.


In [ ]:
df["Calculated Total"] = df["Weight"] + df["Scrap"]
df["Weight Check Difference"] = df["Total Weight"] - df["Calculated Total"]

print("Records where Total Weight does not equal Weight + Scrap:")
display(
    df.loc[
        df["Weight Check Difference"].abs() > 0.0001,
        ["Date", "Weight", "Scrap", "Total Weight", "Weight Check Difference"]
    ]
)

print("All records consistent:", (df["Weight Check Difference"].abs() <= 0.0001).all())


## 3. Overall KPIs

Yahan koi number manually add nahi kiya gaya. Sab totals Excel ke records se calculate ho rahe hain.


In [ ]:
total_records = len(df)
total_weight = df["Weight"].sum()
total_scrap = df["Scrap"].sum()
total_recorded_weight = df["Total Weight"].sum()

# Scrap percentage is calculated against recorded total weight.
overall_scrap_pct = (total_scrap / total_recorded_weight * 100) if total_recorded_weight else 0

kpis = pd.Series({
    "Total Records": total_records,
    "Total Weight": total_weight,
    "Total Scrap": total_scrap,
    "Total Weight (Weight + Scrap)": total_recorded_weight,
    "Overall Scrap %": overall_scrap_pct
})

kpis


## 4. Scrap Analysis

Scrap ko do available components se dekh rahe hain:

- **Seera**
- **Side Cutting**

Kuch rows mein Seera blank hai; un records mein Side Cutting hi Scrap ke barabar ho sakti hai. Blank ko automatically zero treat karna sirf calculation ke liye kiya ja raha hai, source data ko change nahi kiya ja raha.


In [ ]:
scrap_components = pd.DataFrame({
    "Seera": [df["Seera"].sum(skipna=True)],
    "Side Cutting": [df["Side Cutting"].sum(skipna=True)],
    "Total Scrap": [df["Scrap"].sum(skipna=True)]
}).T.rename(columns={0: "Amount"})

scrap_components


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(scrap_components.index, scrap_components["Amount"])
plt.title("Scrap Components")
plt.ylabel("Recorded Amount")
plt.xlabel("Component")
plt.tight_layout()
plt.show()


In [ ]:
df["Scrap %"] = np.where(
    df["Total Weight"] != 0,
    df["Scrap"] / df["Total Weight"] * 100,
    np.nan
)

df[["Date", "Coil", "Weight", "Scrap", "Total Weight", "Scrap %"]].sort_values(
    "Scrap %", ascending=False
).head(10)


## 5. Coil Type Analysis

`Coil` field ke available categories ko compare karenge.

Koi new category manually create nahi ki gayi.


In [ ]:
coil_summary = (
    df.groupby("Coil", dropna=False)
      .agg(
          Records=("Coil", "size"),
          Weight=("Weight", "sum"),
          Scrap=("Scrap", "sum"),
          Total_Weight=("Total Weight", "sum")
      )
      .reset_index()
)

coil_summary["Scrap %"] = np.where(
    coil_summary["Total_Weight"] != 0,
    coil_summary["Scrap"] / coil_summary["Total_Weight"] * 100,
    np.nan
)

coil_summary.sort_values("Total_Weight", ascending=False)


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(coil_summary["Coil"].astype(str), coil_summary["Total_Weight"])
plt.title("Recorded Total Weight by Coil Type")
plt.xlabel("Coil")
plt.ylabel("Total Weight")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 6. Total Size Analysis

`Total Size` ko exactly workbook ke values ke mutabiq group kiya ja raha hai.


In [ ]:
size_summary = (
    df.groupby("Total Size", dropna=False)
      .agg(
          Records=("Total Size", "size"),
          Weight=("Weight", "sum"),
          Scrap=("Scrap", "sum"),
          Total_Weight=("Total Weight", "sum")
      )
      .reset_index()
)

size_summary["Scrap %"] = np.where(
    size_summary["Total_Weight"] != 0,
    size_summary["Scrap"] / size_summary["Total_Weight"] * 100,
    np.nan
)

size_summary.sort_values("Total_Weight", ascending=False)


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(size_summary["Total Size"].astype(str), size_summary["Total_Weight"])
plt.title("Recorded Total Weight by Total Size")
plt.xlabel("Total Size")
plt.ylabel("Total Weight")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 7. Date / Monthly Trend

Monthly trend parsed analysis date par based hai. Agar source mein kisi date ka Excel/text format inconsistent hai, usay **data-quality issue** ke taur par report karna better hai — silently invent/correct karna nahi.


In [ ]:
df["Month"] = df["Analysis Date"].dt.to_period("M").astype(str)

monthly = (
    df.groupby("Month", dropna=False)
      .agg(
          Records=("Month", "size"),
          Weight=("Weight", "sum"),
          Scrap=("Scrap", "sum"),
          Total_Weight=("Total Weight", "sum")
      )
      .reset_index()
)

monthly["Scrap %"] = np.where(
    monthly["Total_Weight"] != 0,
    monthly["Scrap"] / monthly["Total_Weight"] * 100,
    np.nan
)

monthly


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(monthly["Month"], monthly["Total_Weight"], marker="o")
plt.title("Monthly Recorded Total Weight Trend")
plt.xlabel("Month")
plt.ylabel("Total Weight")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(monthly["Month"], monthly["Scrap"], marker="o")
plt.title("Monthly Scrap Trend")
plt.xlabel("Month")
plt.ylabel("Scrap")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Data Quality Findings

Ye section boss ko explain karne ke liye important hai. Analysis ka matlab sirf charts banana nahi; data quality issues identify karna bhi hai.


In [ ]:
date_types = df["Date"].map(type).value_counts()
date_types


In [ ]:
print("Date range in parsed analysis dates:")
print("Min:", df["Analysis Date"].min())
print("Max:", df["Analysis Date"].max())

print("\nRows with Excel/text date storage types:")
print(date_types)

print("\nMissing values:")
display(missing[missing > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTotal Weight consistency:")
print("Incorrect rows:", (df["Weight Check Difference"].abs() > 0.0001).sum())


## 9. Simple Management Summary

Boss ko explain karte waqt is structure ko use kar sakte ho:

- **Total Records:** Excel mein kitni coil-slip entries hain.
- **Total Weight:** recorded `Weight` ka total.
- **Total Scrap:** recorded `Scrap` ka total.
- **Scrap %:** total recorded weight ke against scrap percentage.
- **Coil Analysis:** kis coil type mein kitna recorded weight/scrap hai.
- **Size Analysis:** kis Total Size ka contribution zyada hai.
- **Monthly Trend:** month-wise production/weight aur scrap ka trend.
- **Data Quality:** date formats aur missing values jaisi issues ko identify kiya gaya.

**Important:** Units ko wahi rakha gaya hai jo source Excel mein implied/recorded hain. Jahan unit explicitly nahi di gayi, wahan kg/ton/feet assume nahi kiya gaya.
